In [ ]:
"""
Reduced harmonic-balance solver: mean-field only (mu_a, mu_b), with all
covariances (sigma_a2, sigma_b2, tsigma_a2, tsigma_b2, c, d) FROZEN at fixed
values. Under this freeze, K becomes a plain constant, and P_a, P_a*, P_b, P_b*
drop out entirely -- they only appear in the covariance equations, which are
no longer being solved. This reduces the state from n=14 to n=4:

    dmu_a/dt = -i*wa*mu_a + 2i*EJ*sinep*phia*(exp(-K)*cos(Xbar)-1)
    dmu_b/dt = -i*wb*mu_b - i*epsd*cos(wd t) - kappab/2*mu_b
               + 2i*EJ*sinep*phib*(exp(-K)*cos(Xbar)-1)
    Xbar = phia*(mu_a+mu_a*) + phib*(mu_b+mu_b*)

This is well suited to finding "single coherent state" branches (mu_a != 0,
not the mu_a~0 / large-tsigma_a2 cat branch, which needs the covariances to
be dynamical and self-consistent -- use the full n=14 model for that).

State packed as n=4 real DOFs: y = [Re(mu_a), Im(mu_a), Re(mu_b), Im(mu_b)]
"""

import numpy as np
import sympy as sp
from scipy.sparse import lil_matrix
from scipy.optimize import root
from dataclasses import dataclass


@dataclass
class SnailMeanFieldParams:
    wa: float
    wb: float
    phia: float
    phib: float
    EJ: float
    sinep: float
    epsd: float
    wd: float
    kappab: float
    K: float   # fixed constant, from frozen covariances (see compute_K below)


def compute_K(phia, phib, sigma_a2, tsigma_a2, sigma_b2, tsigma_b2, c, d):
    """Convenience helper: compute K from a set of (possibly guessed) fixed
    covariance values, using the same formula as the full model."""
    return (phia**2 * (sigma_a2 + tsigma_a2.real - 0.5)
            + phib**2 * (sigma_b2 + tsigma_b2.real - 0.5)
            + 2 * phia * phib * (c.real + d.real))


@dataclass
class HBSettingsMeanField:
    N_H: int = 10
    samples_per_harmonic: int = 32
    nu: int = 1
    n: int = 4   # Re(mu_a), Im(mu_a), Re(mu_b), Im(mu_b)

    @property
    def N(self):
        return self.samples_per_harmonic * self.N_H


def _build_symbolic_nonlinear_mf():
    ma_r, ma_i, mb_r, mb_i = sp.symbols('ma_r ma_i mb_r mb_i', real=True)
    phia, phib, EJ, sinep, K = sp.symbols('phia phib EJ sinep K', real=True)
    I = sp.I

    Xbar = 2 * phia * ma_r + 2 * phib * mb_r
    expmK = sp.exp(-K)
    cosX = sp.cos(Xbar)

    dmu_a = 2 * I * EJ * sinep * phia * (expmK * cosX - 1)
    dmu_b = 2 * I * EJ * sinep * phib * (expmK * cosX - 1)

    F = sp.Matrix([sp.re(dmu_a), sp.im(dmu_a), sp.re(dmu_b), sp.im(dmu_b)])
    y = sp.Matrix([ma_r, ma_i, mb_r, mb_i])
    J = F.jacobian(y)

    args = tuple(y) + (phia, phib, EJ, sinep, K)
    F_funcs = [sp.lambdify(args, F[i], modules='numpy') for i in range(4)]
    J_funcs = [[sp.lambdify(args, J[i, j], modules='numpy') for j in range(4)]
               for i in range(4)]
    return F_funcs, J_funcs


_F_SYM_MF, _J_SYM_MF = _build_symbolic_nonlinear_mf()


def _complex_block(c):
    return np.array([[c.real, -c.imag], [c.imag, c.real]])


class SnailMeanFieldHillMethod:
    def __init__(self, mf: SnailMeanFieldParams, settings: HBSettingsMeanField):
        self.p = mf
        self.s = settings
        self.n = settings.n
        self.N_H = settings.N_H
        self.N = settings.N
        self.dim = self.n * (2 * self.N_H + 1)

        self.tau_j = np.linspace(0, 2 * np.pi * settings.nu / mf.wd, self.N, endpoint=False)

        self.Lin = self._build_Lin()
        self.L = self.build_L()
        self.Gamma = self.build_Gamma()
        self.Gamma_pinv = np.linalg.pinv(self.Gamma, rcond=1e-12)
        self.b_ext = self.build_b_ext()

    def _build_Lin(self):
        p = self.p
        Lin = np.zeros((self.n, self.n))
        Lin[0:2, 0:2] = _complex_block(-1j * p.wa)
        Lin[2:4, 2:4] = _complex_block(-1j * p.wb - p.kappab / 2)
        return Lin

    def build_L(self):
        n, N_H, nu = self.n, self.N_H, self.s.nu
        omega = self.p.wd
        dim = self.dim
        L = np.zeros((dim, dim))

        def idx_c0(): return 0
        def idx_sk(k): return 1 + 2 * (k - 1)
        def idx_ck(k): return 2 + 2 * (k - 1)

        i0 = idx_c0()
        L[i0 * n:(i0 + 1) * n, i0 * n:(i0 + 1) * n] = -self.Lin
        for k in range(1, N_H + 1):
            freq = k * omega / nu
            is_, ic_ = idx_sk(k), idx_ck(k)
            L[is_ * n:(is_ + 1) * n, is_ * n:(is_ + 1) * n] = -self.Lin
            L[is_ * n:(is_ + 1) * n, ic_ * n:(ic_ + 1) * n] = -freq * np.eye(n)
            L[ic_ * n:(ic_ + 1) * n, is_ * n:(is_ + 1) * n] = freq * np.eye(n)
            L[ic_ * n:(ic_ + 1) * n, ic_ * n:(ic_ + 1) * n] = -self.Lin
        return L

    def build_Gamma(self):
        t, N_H, n, nu = self.tau_j, self.N_H, self.n, self.s.nu
        omega = self.p.wd
        k = np.arange(1, N_H + 1)
        S = np.sin(np.outer(t, k * omega / nu))
        C = np.cos(np.outer(t, k * omega / nu))
        Phi = np.empty((t.size, 2 * N_H + 1))
        Phi[:, 0] = 1.0 / np.sqrt(2.0)
        Phi[:, 1::2] = S
        Phi[:, 2::2] = C
        return np.kron(Phi, np.eye(n))

    def x_tilde_to_X(self, xt):
        return xt.reshape(self.N, self.n).T

    def X_to_x_tilde(self, X):
        return X.T.reshape(-1)

    def N_time(self, X):
        args_state = list(X)
        Ncol = X.shape[1]
        out = np.empty((self.n, Ncol))
        for i in range(self.n):
            val = _F_SYM_MF[i](*args_state, self.p.phia, self.p.phib, self.p.EJ,
                                self.p.sinep, self.p.K)
            out[i, :] = np.broadcast_to(val, (Ncol,))
        return out

    def dN_dX_blocks(self, X):
        args_state = list(X)
        Ncol = X.shape[1]
        Jarr = np.zeros((self.n, self.n, Ncol))
        for i in range(self.n):
            for j in range(self.n):
                val = _J_SYM_MF[i][j](*args_state, self.p.phia, self.p.phib,
                                       self.p.EJ, self.p.sinep, self.p.K)
                Jarr[i, j, :] = np.broadcast_to(val, (Ncol,))
        return [Jarr[:, :, k] for k in range(Ncol)]

    def build_dNtilde_dx_tilde(self, X):
        blocks = self.dN_dX_blocks(X)
        Jbig = lil_matrix((self.n * self.N, self.n * self.N))
        for j, Jj in enumerate(blocks):
            rows = slice(j * self.n, (j + 1) * self.n)
            Jbig[rows, rows] = Jj
        return Jbig.tocsr()

    def b_nl(self, z):
        X = self.x_tilde_to_X(self.Gamma @ z)
        Ftime = self.N_time(X)
        return self.Gamma_pinv @ self.X_to_x_tilde(Ftime)

    def db_dz(self, z):
        X = self.x_tilde_to_X(self.Gamma @ z)
        Jbig = self.build_dNtilde_dx_tilde(X)
        return self.Gamma_pinv @ (Jbig @ self.Gamma)

    def build_b_ext(self):
        n, N_H = self.n, self.N_H
        b = np.zeros(self.dim)

        def idx_ck(k): return 2 + 2 * (k - 1)

        if N_H >= self.s.nu:
            icn = idx_ck(self.s.nu)
            b[icn * n + 3] += -self.p.epsd
        return b

    def residual(self, z):
        return self.L @ z - self.b_nl(z) - self.b_ext

    def jacobian(self, z):
        return self.L - self.db_dz(z)

    def solve(self, z0, **kwargs):
        sol = root(self.residual, z0, jac=self.jacobian, method='hybr', **kwargs)
        if not sol.success:
            raise RuntimeError(f"HB solve did not converge: {sol.message}")
        return sol.x

In [ ]:
import numpy as np
from scipy.special import jv
import matplotlib.pyplot as plt
w_a = 25.338776456203686
w_b = 2 * w_a
phi_a, phi_b = 0.11, 0.204
E_J = 37.12 * 2 * np.pi
kappa_b = 3 / 10.4
w_d = w_b
epsilon_p = 0.23
g_bessel = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
epsilon_d = 6.12 * g_bessel
sinep = np.sin(epsilon_p)

# guess K from near-vacuum covariances (sigma~1, tsigma~0, c=d=0)
K0 = compute_K(phi_a, phi_b, 1.0, .5j, 1.0, .2+.2j, .2+.2j, .2+.2j)
print('K (vacuum guess):', K0)

p = SnailMeanFieldParams(wa=w_a, wb=w_b, phia=phi_a, phib=phi_b, EJ=E_J, sinep=sinep,
                          epsd=epsilon_d, wd=w_d, kappab=kappa_b, K=K0)
s = HBSettingsMeanField(N_H=6, samples_per_harmonic=900, nu=2)
m = SnailMeanFieldHillMethod(p, s)
print('dim =', m.dim)

# try several initial guesses to look for nontrivial (mu_a != 0) branches
g = 2 * E_J * sinep * phi_a**2 * phi_b
pred_amp = np.sqrt(epsilon_d / g)
print('predicted cat amplitude scale sqrt(epsd/g):', pred_amp)

z0 = np.zeros(m.dim)
try:
    sol = m.solve(z0)
    X_hb = m.x_tilde_to_X(m.Gamma @ sol)
    mu_a = X_hb[0] + 1j * X_hb[1]
    mu_b = X_hb[2] + 1j * X_hb[3]
    resid = np.linalg.norm(m.residual(sol))
    print(f" converged  "
            f"|mu_a| mean={np.mean(np.abs(mu_a)):.4f}  "
            f"|mu_b| mean={np.mean(np.abs(mu_b)):.4f}  resid={resid:.2e}")
except RuntimeError as e:
    print(f" failed: {e}")

plt.plot(mu_a.real)
plt.plot(mu_a.imag)
plt.show()

In [ ]:
plt.plot(mu_b.real)
plt.plot(mu_b.imag)
plt.show()